### 0. read and check

In [3]:
import os
import pandas as pd

data_dir = "/home/lfj/projects_dir/MERF/baselines/Venus-MAXWELL/example_datasets"

for split in ["train", "valid", "test"]:
    csv_dir = os.path.join(data_dir, split, 'mutant')
    csv_list = os.listdir(csv_dir)

    dataset_num = len(csv_list)
    sample_count = 0

    for csv_file in csv_list:
        assert csv_file.endswith('.csv')
        df = pd.read_csv(os.path.join(csv_dir, csv_file))
        sample_count += len(df)
    
    print(f"{split} set: {dataset_num} proteins, {sample_count} samples in total.")

train set: 229 proteins, 201690 samples in total.
valid set: 69 proteins, 62153 samples in total.
test set: 308 proteins, 12410 samples in total.


In [5]:
# check scripts in vm
import os
from pathlib import Path
from Bio import SeqIO
import esm.inverse_folding
import pandas as pd

def read_sequence_from_fasta(fasta_file):
    for record in SeqIO.parse(fasta_file, "fasta"):
        return str(record.seq)

def read_sequence_from_pdb(pdb_file):
    _, native_seq = esm.inverse_folding.util.load_coords(str(pdb_file), "A")
    return native_seq

def read_mutants_from_csv(mutant_file):
    df = pd.read_csv(mutant_file)
    return df["mutant"].tolist()

def check_dataset(dataset_path):
    dataset_path = Path(dataset_path)
    mutant_path = dataset_path / "mutant"
    fasta_path = dataset_path / "fasta"
    pdb_path = dataset_path / "pdb"
    mutant_files = list(mutant_path.glob("*.csv"))
    names = [each.stem for each in mutant_files]
    for name in names:
        # Check fasta file
        fasta_file = fasta_path / f"{name}.fasta"
        if not fasta_file.exists():
            raise FileNotFoundError(f"Fasta file {fasta_file} does not exist")
        fasta_seq = read_sequence_from_fasta(fasta_file)
        
        pdb_file = pdb_path / f"{name}.pdb"
        if not pdb_file.exists():
            raise FileNotFoundError(f"PDB file {pdb_file} does not exist")
        pdb_seq = read_sequence_from_pdb(pdb_file)
        
        mutant_file = mutant_path / f"{name}.csv"
        if not mutant_file.exists():
            raise FileNotFoundError(f"Mutant file {mutant_file} does not exist")
        mutants = read_mutants_from_csv(mutant_file)
        
        # Check sequence consistency
        if fasta_seq != pdb_seq:
            raise ValueError(f"Sequence mismatch between fasta and pdb for {name}")

        # Check mutant consistency
        for mutant in mutants:
            if ":" in mutant or ";" in mutant:
                raise ValueError(f"Invalid mutant {mutant} in {mutant_file} (Only single amino acid changes are allowed)")
            wt, idx, mut = mutant[0], int(mutant[1:-1]) - 1, mutant[-1]
            if idx < 0 or idx >= len(fasta_seq):
                raise ValueError(f"Invalid mutant {mutant} in {mutant_file} (Index out of bounds)")
            if fasta_seq[idx] != wt:
                raise ValueError(f"Invalid mutant {mutant} in {mutant_file} (Wildtype mismatch with seq in {fasta_file})")
        
        print(f"Dataset {name} passed all checks")

data_dir = "/home/lfj/projects_dir/MERF/baselines/Venus-MAXWELL/example_datasets"

for split in ["train", "valid", "test"]:
    dataset_path = os.path.join(data_dir, split)
    print(f"Checking {split} dataset...")
    check_dataset(dataset_path)

Checking train dataset...
Dataset 1A32 passed all checks
Dataset 1AOY passed all checks
Dataset 1E0L passed all checks
Dataset 1ENH passed all checks
Dataset 1F0M passed all checks
Dataset 1GJS passed all checks
Dataset 1GYZ passed all checks
Dataset 1I2T passed all checks
Dataset 1I6C passed all checks
Dataset 1IFY passed all checks
Dataset 1K1V passed all checks
Dataset 1LP1 passed all checks
Dataset 1PSE passed all checks
Dataset 1PV0 passed all checks
Dataset 1QP2 passed all checks
Dataset 1R69 passed all checks
Dataset 1TG0 passed all checks
Dataset 1UFM passed all checks
Dataset 1URF passed all checks
Dataset 1V1C passed all checks
Dataset 1VII passed all checks
Dataset 1W4F passed all checks
Dataset 1W4G passed all checks
Dataset 1W4H passed all checks
Dataset 1WR4 passed all checks
Dataset 1Y0M passed all checks
Dataset 1YRF passed all checks
Dataset 1YU5 passed all checks
Dataset 2B88 passed all checks
Dataset 2B89 passed all checks
Dataset 2BTH passed all checks
Dataset 2CJJ 

### 1. process to transform into our data format

In [2]:
import os
import pandas as pd
from pathlib import Path
from Bio import SeqIO
import esm.inverse_folding
import pandas as pd

In [8]:
# add chain info to the origin csv files
# 感觉全是A，检查一下是不是这样？

data_dir = "/home/lfj/projects_dir/MERF/baselines/Venus-MAXWELL/example_datasets"

for split in ["train", "valid", "test"]:
    dataset_path = os.path.join(data_dir, split)

    dataset_path = Path(dataset_path)
    pdb_path = dataset_path / "pdb"
    pdb_files = list(pdb_path.glob("*.pdb"))

    for pdb_file in pdb_files:
        name = pdb_file.stem
        with open(pdb_file, 'r') as f:
            lines = f.readlines()
        chains = set()
        for line in lines:
            if line.startswith("ATOM"):
                chain_id = line[21].strip()
                chains.add(chain_id)
        assert chains == {'A'}, f"More than one chain found in {pdb_file}"
        print(f"{split}_{name}: chain A confirmed.")

train_1A32: chain A confirmed.
train_1AOY: chain A confirmed.
train_1E0L: chain A confirmed.
train_1ENH: chain A confirmed.
train_1F0M: chain A confirmed.
train_1GJS: chain A confirmed.
train_1GYZ: chain A confirmed.
train_1I2T: chain A confirmed.
train_1I6C: chain A confirmed.
train_1IFY: chain A confirmed.
train_1K1V: chain A confirmed.
train_1LP1: chain A confirmed.
train_1PSE: chain A confirmed.
train_1PV0: chain A confirmed.
train_1QP2: chain A confirmed.
train_1R69: chain A confirmed.
train_1TG0: chain A confirmed.
train_1UFM: chain A confirmed.
train_1URF: chain A confirmed.
train_1V1C: chain A confirmed.
train_1VII: chain A confirmed.
train_1W4F: chain A confirmed.
train_1W4G: chain A confirmed.
train_1W4H: chain A confirmed.
train_1WR4: chain A confirmed.
train_1Y0M: chain A confirmed.
train_1YRF: chain A confirmed.
train_1YU5: chain A confirmed.
train_2B88: chain A confirmed.
train_2B89: chain A confirmed.
train_2BTH: chain A confirmed.
train_2CJJ: chain A confirmed.
train_2G

In [ ]:
# gather all data in one dataframe for each dataset

src_dir = '/home/lfj/projects_dir/MERF/baselines/Venus-MAXWELL/example_datasets'
target_dir = '/home/lfj/projects_dir/MERF/data/VenusMaxwell'

for split in ["train", "valid", "test"]:
    pdb_src_path = os.path.join(src_dir, split, 'pdb')
    csv_file_list = os.listdir(os.path.join(src_dir, split, 'mutant'))

    csv_target_path = os.path.join(target_dir, f"{split}_data.csv")
    pdb_target_path = os.path.join(target_dir, "PDBs")

    pdb_id_list = []
    chain_id_list = []
    mutant_list = []
    ddg_list = []

    for csv_file in csv_file_list:

        pdb_id = csv_file[:-4]
        chain_id = 'A'

        data_df = pd.read_csv(os.path.join(src_dir, split, 'mutant', csv_file))

        mutants = data_df['mutant'].tolist()
        ddgs = data_df['ddG'].tolist()

        pdb_id_list.extend([pdb_id] * len(mutants))
        chain_id_list.extend([chain_id] * len(mutants))
        mutant_list.extend(mutants)
        ddg_list.extend(ddgs)
    
    final_df = pd.DataFrame({
        'pdb_id': pdb_id_list,
        'chain_id': chain_id_list,
        'mutant': mutant_list,
        'ddG': ddg_list
    })

    final_df.to_csv(csv_target_path, index=False)
    print(f"Saved {split} data to {csv_target_path}")

    os.makedirs(pdb_target_path, exist_ok=True)
    for pdb_file in os.listdir(pdb_src_path):
        src_file = os.path.join(pdb_src_path, pdb_file)
        dst_file = os.path.join(pdb_target_path, pdb_file)
        os.system(f"cp {src_file} {dst_file}")
    print(f"Copied PDB files to {pdb_target_path}")

Saved train data to /home/lfj/projects_dir/MERF/data/VenusMaxwell/train_data.csv
Copied PDB files to /home/lfj/projects_dir/MERF/data/VenusMaxwell/PDBs_fixed
Saved valid data to /home/lfj/projects_dir/MERF/data/VenusMaxwell/valid_data.csv
Copied PDB files to /home/lfj/projects_dir/MERF/data/VenusMaxwell/PDBs_fixed
Saved test data to /home/lfj/projects_dir/MERF/data/VenusMaxwell/test_data.csv
Copied PDB files to /home/lfj/projects_dir/MERF/data/VenusMaxwell/PDBs_fixed


In [ ]:
# 剔除掉数据中所有自己突变成自己的sample

for split in ["train", "valid", "test"]:
    csv_file = f"/home/lfj/projects_dir/MERF/data/VenusMaxwell/{split}_data.csv"
    df = pd.read_csv(csv_file)

    def is_self_mutation(row):
        mutant = row['mutant']
        wt = mutant[0]
        mut = mutant[-1]
        return wt == mut

    initial_count = len(df)
    df_filtered = df[~df.apply(is_self_mutation, axis=1)]
    final_count = len(df_filtered)

    df_filtered.to_csv(csv_file, index=False)
    print(f"{split} set: Removed {initial_count - final_count} self-mutation samples, {final_count} samples remain.")

train set: Removed 1 self-mutation samples, 1195 samples remain.


FileNotFoundError: [Errno 2] No such file or directory: '/home/lfj/projects_dir/MERF/data/VenusMaxwell/valid_data_1A32.csv'

### 2. Gen mutant

见scripts/mutate_vm.py

In [1]:
# 检查是不是都突变完了

import os
import pandas as pd
from pathlib import Path
from Bio import SeqIO
import esm.inverse_folding
import pandas as pd
from tqdm import tqdm

data_dir = "/home/lfj/projects_dir/MERF/data/VenusMaxwell"
wt_dir = "/home/lfj/projects_dir/MERF/data/VenusMaxwell/PDBs"
fix_dir = "/home/lfj/projects_dir/MERF/data/VenusMaxwell/PDBs_fixed"
mut_dir = "/home/lfj/projects_dir/MERF/data/VenusMaxwell/PDBs_mutated"

# Combine all splits
dfs = []
for split in ['train', 'valid', 'test']:
    data_path = os.path.join(data_dir, f"{split}_data.csv")
    dfs.append(pd.read_csv(data_path))

df_all = pd.concat(dfs, ignore_index=True)

for idx in tqdm(range(len(df_all))):
    row = df_all.iloc[idx]
    pdb_id = row['pdb_id']
    chain_id = row['chain_id']
    mutant = row['mutant']

    mutate_info_new = []
    for mut in mutant.split(','):
        wtname = mut[0]
        resid = str(int(mut[1:-1]))
        mutname = mut[-1]
        mutate_info_reformat = f"{wtname}{chain_id}{resid}{mutname}"
        mutate_info_new.append(mutate_info_reformat)
    mutate_info_new = ','.join(mutate_info_new)

    wt_pdb_file = os.path.join(wt_dir, f"{pdb_id}.pdb")
    fix_pdb_file = os.path.join(fix_dir, f"{pdb_id}.pdb")
    mut_pdb_file = os.path.join(mut_dir, f"{pdb_id}_{mutate_info_new}.pdb")

    if not os.path.exists(fix_pdb_file):
        raise FileNotFoundError(f"Fixed PDB file {fix_pdb_file} does not exist")
    if not os.path.exists(mut_pdb_file):
        raise FileNotFoundError(f"Mutated PDB file {mut_pdb_file} does not exist")


100%|██████████| 275741/275741 [00:25<00:00, 10841.64it/s]


In [ ]:
# （legacy，使用下方并行版）检查突变后的位点是否为想要的突变结果
import os
import pandas as pd
from tqdm import tqdm
from Bio.PDB import PDBParser, PDBIO, Select, Selection

STANDARD_RESIDUE_SUBSTITUTIONS = {
    '2AS':'ASP', '3AH':'HIS', '5HP':'GLU', 'ACL':'ARG', 'AGM':'ARG', 'AIB':'ALA', 'ALM':'ALA',
    'ALA':'ALA', 'ARG':'ARG', 'ASN':'ASN', 'ASP':'ASP', 'CYS':'CYS', 'GLU':'GLU', 'GLN':'GLN',
    'GLY':'GLY', 'HIS':'HIS', 'ILE':'ILE', 'LEU':'LEU', 'LYS':'LYS', 'MET':'MET', 'PHE':'PHE',
    'PRO':'PRO', 'SER':'SER', 'THR':'THR', 'TRP':'TRP', 'TYR':'TYR', 'VAL':'VAL', 'UNK':'UNK', 'MSE':'MET'
}

RESIDUE_NAME_TO_TOKEN = {
    "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C",
    "GLU": "E", "GLN": "Q", "GLY": "G", "HIS": "H", "ILE": "I",
    "LEU": "L", "LYS": "K", "MET": "M", "PHE": "F", "PRO": "P",
    "SER": "S", "THR": "T", "TRP": "W", "TYR": "Y", "VAL": "V", "UNK": "X",
}

def is_aa(value):
    return value in STANDARD_RESIDUE_SUBSTITUTIONS

def parse_biopython_structure_mapping(pdb_id, entity, max_seq_len=None):
    chains = Selection.unfold_entities(entity, 'C')
    chains.sort(key=lambda c: c.get_id())
    data = {}

    assert len(chains) == 1, f"Expected one chain in antigen entity for {pdb_id}"

    pos_aa_mapping = {}

    for chain in chains:
        residues = Selection.unfold_entities(chain, 'R')
        residues.sort(key=lambda res: (res.get_id()[1], res.get_id()[2]))
        
        for res in residues:
            res_id = int(res.get_id()[1])
            if max_seq_len is not None and res_id > max_seq_len:
                break

            resname = res.get_resname()
            if not is_aa(resname):
                continue

            std_name = STANDARD_RESIDUE_SUBSTITUTIONS[resname]
            token = RESIDUE_NAME_TO_TOKEN[std_name]

            pos_aa_mapping[res_id] = token

    return pos_aa_mapping

# 1. read data and set path
data_dir = "/home/lfj/projects_dir/MERF/data/VenusMaxwell"
dfs = []
for split in ['train', 'valid', 'test']:
    data_path = os.path.join(data_dir, f"{split}_data.csv")
    dfs.append(pd.read_csv(data_path))
data_df = pd.concat(dfs, ignore_index=True)

mut_dir = "/home/lfj/projects_dir/MERF/data/VenusMaxwell/PDBs_mutated"
# 1. end

for idx in tqdm(range(len(data_df))):

    # 2. parse mutation info
    pdb_id = data_df.loc[idx, 'pdb_id']
    chain_id = data_df.loc[idx, 'chain_id']
    mutant = data_df.loc[idx, 'mutant']

    mutate_info = f"{mutant[0]}{chain_id}{mutant[1:-1]}{mutant[-1]}"
    # 2. end

    pdb_path = os.path.join(mut_dir, f"{pdb_id}_{mutate_info}.pdb")
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(pdb_id, pdb_path)
    pdb_structure = structure[0]

    pos_aa_mapping = parse_biopython_structure_mapping(pdb_id, pdb_structure[chain_id])

    # 3. check mutate info

    if pos_aa_mapping.get(int(mutant[1:-1]), '') != mutant[-1]:
        print(f"Mutation position mismatch in {pdb_id} for mutation {mutant}: expected {mutant[-1]}, found {pos_aa_mapping.get(int(mutant[1:-1]), '')}")


In [1]:
# 检查突变后的位点是否为想要的突变结果
import os
import pandas as pd
from tqdm import tqdm
from multiprocessing import Pool
from Bio.PDB import PDBParser, PDBIO, Select, Selection

STANDARD_RESIDUE_SUBSTITUTIONS = {
    '2AS':'ASP', '3AH':'HIS', '5HP':'GLU', 'ACL':'ARG', 'AGM':'ARG', 'AIB':'ALA', 'ALM':'ALA',
    'ALA':'ALA', 'ARG':'ARG', 'ASN':'ASN', 'ASP':'ASP', 'CYS':'CYS', 'GLU':'GLU', 'GLN':'GLN',
    'GLY':'GLY', 'HIS':'HIS', 'ILE':'ILE', 'LEU':'LEU', 'LYS':'LYS', 'MET':'MET', 'PHE':'PHE',
    'PRO':'PRO', 'SER':'SER', 'THR':'THR', 'TRP':'TRP', 'TYR':'TYR', 'VAL':'VAL', 'UNK':'UNK', 'MSE':'MET'
}

RESIDUE_NAME_TO_TOKEN = {
    "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C",
    "GLU": "E", "GLN": "Q", "GLY": "G", "HIS": "H", "ILE": "I",
    "LEU": "L", "LYS": "K", "MET": "M", "PHE": "F", "PRO": "P",
    "SER": "S", "THR": "T", "TRP": "W", "TYR": "Y", "VAL": "V", "UNK": "X",
}

def is_aa(value):
    return value in STANDARD_RESIDUE_SUBSTITUTIONS

def parse_biopython_structure_mapping(pdb_id, entity, max_seq_len=None):
    chains = Selection.unfold_entities(entity, 'C')
    chains.sort(key=lambda c: c.get_id())
    data = {}

    assert len(chains) == 1, f"Expected one chain in antigen entity for {pdb_id}"

    pos_aa_mapping = {}

    for chain in chains:
        residues = Selection.unfold_entities(chain, 'R')
        residues.sort(key=lambda res: (res.get_id()[1], res.get_id()[2]))

        for res in residues:
            res_id = int(res.get_id()[1])
            if max_seq_len is not None and res_id > max_seq_len:
                break

            resname = res.get_resname()
            if not is_aa(resname):
                continue

            std_name = STANDARD_RESIDUE_SUBSTITUTIONS[resname]
            token = RESIDUE_NAME_TO_TOKEN[std_name]

            pos_aa_mapping[res_id] = token

    return pos_aa_mapping

def check_mutation(args):
    """Check mutation for a single row"""
    idx, pdb_id, chain_id, mutant, mut_dir = args

    # 2. update mutate_info
    mutate_info = f"{mutant[0]}{chain_id}{mutant[1:-1]}{mutant[-1]}"
    pdb_path = os.path.join(mut_dir, f"{pdb_id}_{mutate_info}.pdb")
    # 3. end

    try:
        parser = PDBParser(QUIET=True)
        structure = parser.get_structure(pdb_id, pdb_path)
        pdb_structure = structure[0]

        pos_aa_mapping = parse_biopython_structure_mapping(pdb_id, pdb_structure[chain_id])

        # Check mutate info
        if pos_aa_mapping.get(int(mutant[1:-1]), '') != mutant[-1]:
            error_msg = f"Mutation position mismatch in {pdb_id} for mutation {mutant}: expected {mutant[-1]}, found {pos_aa_mapping.get(int(mutant[1:-1]), '')}"
            return error_msg
    except Exception as e:
        error_msg = f"Error processing {pdb_id} for mutation {mutant}: {str(e)}"
        return error_msg

    return None

# 1. read data and set path
data_dir = "/home/lfj/projects_dir/MERF/data/VenusMaxwell"
dfs = []
for split in ['train', 'valid', 'test']:
    data_path = os.path.join(data_dir, f"{split}_data.csv")
    dfs.append(pd.read_csv(data_path))
data_df = pd.concat(dfs, ignore_index=True)

mut_dir = "/home/lfj/projects_dir/MERF/data/VenusMaxwell/PDBs_mutated"
# 1. end

# 2. parse mutation info
args_list = [
    (idx, data_df.loc[idx, 'pdb_id'], data_df.loc[idx, 'chain_id'], data_df.loc[idx, 'mutant'], mut_dir)
    for idx in range(len(data_df))
]
# 2. end

# 并行处理（使用 tqdm 显示进度）
with Pool(processes=16) as pool:
    results = list(tqdm(
        pool.imap_unordered(check_mutation, args_list),
        total=len(args_list),
        desc="Checking mutations"
    ))

# 收集并打印错误信息
errors = [msg for msg in results if msg is not None]

if errors:
    print(f"\nFound {len(errors)} mutation errors:")
    for error in errors:
        print(error)
else:
    print(f"\nAll {len(data_df)} mutations are correct!")


Checking mutations: 100%|██████████| 275741/275741 [02:52<00:00, 1600.40it/s]



All 275741 mutations are correct!


### 3. check SK2

见scripts/mutate_sk.py

有两个手动处理，2B2X和1KBH

In [12]:
# 删了1KBH
import os
import pandas as pd

data_path = "/home/lfj/projects_dir/MERF/data/SKEMPIv2/SKEMPIv2_old.csv"
df = pd.read_csv(data_path)
df = df[df['pdb_id'] != '1KBH']
df.to_csv("/home/lfj/projects_dir/MERF/data/SKEMPIv2/SKEMPIv2.csv", index=False)

In [14]:
# 检查sk2是不是都突变完了

import os
import pandas as pd
from pathlib import Path
from Bio import SeqIO
import esm.inverse_folding
import pandas as pd
from tqdm import tqdm

data_path = "/home/lfj/projects_dir/MERF/data/SKEMPIv2/SKEMPIv2.csv"
wt_dir = "/home/lfj/projects_dir/MERF/data/SKEMPIv2/PDBs"
fix_dir = "/home/lfj/projects_dir/MERF/data/SKEMPIv2/PDBs_fixed"
mut_dir = "/home/lfj/projects_dir/MERF/data/SKEMPIv2/PDBs_mutated"

df_all = pd.read_csv(data_path)

for idx in tqdm(range(len(df_all))):
    row = df_all.iloc[idx]
    pdb_id = row['pdb_id']
    pdb_id = pdb_id.replace('+', '')
    pdb_id = pdb_id.replace('.00', '')
    mutant = row['mutant']

    mutate_info_new = mutant

    wt_pdb_file = os.path.join(wt_dir, f"{pdb_id}.pdb")
    fix_pdb_file = os.path.join(fix_dir, f"{pdb_id}.pdb")
    mut_pdb_file = os.path.join(mut_dir, f"{pdb_id}_{mutate_info_new}.pdb")

    if not os.path.exists(fix_pdb_file):
        raise FileNotFoundError(f"Fixed PDB file {fix_pdb_file} does not exist")
    if not os.path.exists(mut_pdb_file):
        print(f"Mutated PDB file {mut_pdb_file} does not exist")


100%|██████████| 5783/5783 [00:00<00:00, 18245.20it/s]


In [2]:
# 检查突变后的位点是否为想要的突变结果
import os
import pandas as pd
from tqdm import tqdm
from multiprocessing import Pool
from Bio.PDB import PDBParser, PDBIO, Select, Selection

STANDARD_RESIDUE_SUBSTITUTIONS = {
    '2AS':'ASP', '3AH':'HIS', '5HP':'GLU', 'ACL':'ARG', 'AGM':'ARG', 'AIB':'ALA', 'ALM':'ALA',
    'ALA':'ALA', 'ARG':'ARG', 'ASN':'ASN', 'ASP':'ASP', 'CYS':'CYS', 'GLU':'GLU', 'GLN':'GLN',
    'GLY':'GLY', 'HIS':'HIS', 'ILE':'ILE', 'LEU':'LEU', 'LYS':'LYS', 'MET':'MET', 'PHE':'PHE',
    'PRO':'PRO', 'SER':'SER', 'THR':'THR', 'TRP':'TRP', 'TYR':'TYR', 'VAL':'VAL', 'UNK':'UNK', 'MSE':'MET'
}

RESIDUE_NAME_TO_TOKEN = {
    "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C",
    "GLU": "E", "GLN": "Q", "GLY": "G", "HIS": "H", "ILE": "I",
    "LEU": "L", "LYS": "K", "MET": "M", "PHE": "F", "PRO": "P",
    "SER": "S", "THR": "T", "TRP": "W", "TYR": "Y", "VAL": "V", "UNK": "X",
}

def is_aa(value):
    return value in STANDARD_RESIDUE_SUBSTITUTIONS

def parse_biopython_structure_mapping(pdb_id, entity, max_seq_len=None):
    chains = Selection.unfold_entities(entity, 'C')
    chains.sort(key=lambda c: c.get_id())
    data = {}

    assert len(chains) == 1, f"Expected one chain in antigen entity for {pdb_id}"

    pos_aa_mapping = {}

    for chain in chains:
        residues = Selection.unfold_entities(chain, 'R')
        residues.sort(key=lambda res: (res.get_id()[1], res.get_id()[2]))

        for res in residues:
            res_id = int(res.get_id()[1])
            if max_seq_len is not None and res_id > max_seq_len:
                break

            resname = res.get_resname()
            if not is_aa(resname):
                continue

            std_name = STANDARD_RESIDUE_SUBSTITUTIONS[resname]
            token = RESIDUE_NAME_TO_TOKEN[std_name]

            pos_aa_mapping[res_id] = token

    return pos_aa_mapping

def check_mutation(args):
    """Check mutation for a single row"""
    idx, pdb_id, mutant, mut_dir = args

    pdb_id = pdb_id.replace('+', '')
    pdb_id = pdb_id.replace('.00', '')

    # 2. update mutate_info
    mutate_infos = mutant
    pdb_path = os.path.join(mut_dir, f"{pdb_id}_{mutate_infos}.pdb")
    # 3. end

    try:
        parser = PDBParser(QUIET=True)
        structure = parser.get_structure(pdb_id, pdb_path)
        pdb_structure = structure[0]

        for mutate_info in mutate_infos.split(','):
            chain_id = mutate_info[1]

            pos_aa_mapping = parse_biopython_structure_mapping(pdb_id, pdb_structure[chain_id])

            # Check mutate info
            if pos_aa_mapping.get(int(mutate_info[2:-1]), '') != mutate_info[-1]:
                error_msg = f"Mutation position mismatch in {pdb_id} for mutation {mutate_info}: expected {mutate_info[-1]}, found {pos_aa_mapping.get(int(mutate_info[1:-1]), '')}"
                return error_msg
    except Exception as e:
        error_msg = f"Error processing {pdb_id} for mutation {mutant}: {str(e)}"
        return error_msg

    return None

# 1. read data and set path
data_path = "/home/lfj/projects_dir/MERF/data/SKEMPIv2/SKEMPIv2.csv"
data_df = pd.read_csv(data_path)

mut_dir = "/home/lfj/projects_dir/MERF/data/SKEMPIv2/PDBs_mutated"

# 1. end

# 2. parse mutation info
args_list = [
    (idx, data_df.loc[idx, 'pdb_id'], data_df.loc[idx, 'mutant'], mut_dir)
    for idx in range(len(data_df))
]
# 2. end

# 并行处理（使用 tqdm 显示进度）
with Pool(processes=16) as pool:
    results = list(tqdm(
        pool.imap_unordered(check_mutation, args_list),
        total=len(args_list),
        desc="Checking mutations"
    ))

# 收集并打印错误信息
errors = [msg for msg in results if msg is not None]

if errors:
    print(f"\nFound {len(errors)} mutation errors:")
    for error in errors:
        print(error)
else:
    print(f"\nAll {len(data_df)} mutations are correct!")


Checking mutations: 100%|██████████| 5783/5783 [00:29<00:00, 197.69it/s]


All 5783 mutations are correct!


### 4. check abbind，其实也应该放到test set

In [1]:
# 看下哪些pdbid的突变信息信息中出现了icode

import pandas as pd

data_path = "/home/lfj/projects_dir/MERF/data/ABbind/AB-Bind_645pMulti_old.csv"
df = pd.read_csv(data_path)

pdb_id_list = []

for idx in range(len(df)):
    row = df.iloc[idx]
    pdb_id = row['pdb_id']
    mutant = row['mutant']
    for mut in mutant.split(','):
        resid = str(mut[3:-1])
        if not resid.isdigit():
            # print(f"PDB ID {pdb_id} has icode in mutant {mut}")
            pdb_id_list.append(pdb_id)

print("PDB IDs with icode in mutants:", set(pdb_id_list))

PDB IDs with icode in mutants: {'3BN9', '3BDY', '3BE1', '2NY7', '1N8Z', 'HM_3BN9'}


In [ ]:
# 尝试重新编号pdb并生成映射文件，见脚本

In [4]:
# 根据重新编号的pdb，修改csv，同时把非foldX的突变信息和换掉，修改后确保读入什么就突变什么也存储什么
import pandas as pd
from tqdm import tqdm
import json

RENUMBER_PDB_IDS = ['2NY7', '3BE1', 'HM_3BN9', '3BN9', '3BDY', '1N8Z']

data_path = "/home/lfj/projects_dir/MERF/data/ABbind/AB-Bind_645pMulti.csv"
df = pd.read_csv(data_path)

new_mutant_list = []

for idx in range(len(df)):
    pdb_id = df.iloc[idx]['pdb_id']
    mutate_info = df.iloc[idx]['mutant']

    if pdb_id in RENUMBER_PDB_IDS:
        mapping_file = os.path.join("/home/lfj/projects_dir/MERF/data/ABbind/PDBs_renumbered", f"{pdb_id}_residue_mapping.json")
        with open(mapping_file, 'r') as f:
            residue_mapping = json.load(f)

    mutate_info_new = []
    for mut in mutate_info.split(','):
        chain_id = mut[0]
        wtname = mut[2]
        resid = str(mut[3:-1])
        mutname = mut[-1]

        if pdb_id in RENUMBER_PDB_IDS:
            if resid.isdigit():
                resid = residue_mapping[chain_id][f'{resid}_ ']
            else:
                resid  = residue_mapping[chain_id][f"{resid[:-1]}_{resid[-1].upper()}"]

        mutate_info_reformat = f"{wtname}{chain_id}{resid}{mutname}"
        mutate_info_new.append(mutate_info_reformat)
    mutate_info_new = ','.join(mutate_info_new)

    new_mutant_list.append(mutate_info_new)

df['mutant'] = new_mutant_list
df.to_csv(data_path, index=False)

In [6]:
# 检查abbind是不是都突变完了

import os
import pandas as pd
from pathlib import Path
from Bio import SeqIO
import esm.inverse_folding
import pandas as pd
from tqdm import tqdm

data_path = "/home/lfj/projects_dir/MERF/data/ABbind/AB-Bind_645pMulti.csv"
wt_dir = "/home/lfj/projects_dir/MERF/data/ABbind/PDBs"
fix_dir = "/home/lfj/projects_dir/MERF/data/ABbind/PDBs_fixed"
mut_dir = "/home/lfj/projects_dir/MERF/data/ABbind/PDBs_mutated"

df_all = pd.read_csv(data_path)

for idx in tqdm(range(len(df_all))):
    row = df_all.iloc[idx]
    pdb_id = row['pdb_id']
    mutant = row['mutant']

    mutate_info_new = mutant

    wt_pdb_file = os.path.join(wt_dir, f"{pdb_id}.pdb")
    fix_pdb_file = os.path.join(fix_dir, f"{pdb_id}.pdb")
    mut_pdb_file = os.path.join(mut_dir, f"{pdb_id}_{mutate_info_new}.pdb")

    if not os.path.exists(fix_pdb_file):
        raise FileNotFoundError(f"Fixed PDB file {fix_pdb_file} does not exist")
    if not os.path.exists(mut_pdb_file):
        print(f"Mutated PDB file {mut_pdb_file} does not exist")


100%|██████████| 1095/1095 [00:00<00:00, 11989.92it/s]


In [5]:
# 检查突变后的位点是否为想要的突变结果
import os
import pandas as pd
from tqdm import tqdm
from multiprocessing import Pool
from Bio.PDB import PDBParser, PDBIO, Select, Selection

STANDARD_RESIDUE_SUBSTITUTIONS = {
    '2AS':'ASP', '3AH':'HIS', '5HP':'GLU', 'ACL':'ARG', 'AGM':'ARG', 'AIB':'ALA', 'ALM':'ALA',
    'ALA':'ALA', 'ARG':'ARG', 'ASN':'ASN', 'ASP':'ASP', 'CYS':'CYS', 'GLU':'GLU', 'GLN':'GLN',
    'GLY':'GLY', 'HIS':'HIS', 'ILE':'ILE', 'LEU':'LEU', 'LYS':'LYS', 'MET':'MET', 'PHE':'PHE',
    'PRO':'PRO', 'SER':'SER', 'THR':'THR', 'TRP':'TRP', 'TYR':'TYR', 'VAL':'VAL', 'UNK':'UNK', 'MSE':'MET'
}

RESIDUE_NAME_TO_TOKEN = {
    "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C",
    "GLU": "E", "GLN": "Q", "GLY": "G", "HIS": "H", "ILE": "I",
    "LEU": "L", "LYS": "K", "MET": "M", "PHE": "F", "PRO": "P",
    "SER": "S", "THR": "T", "TRP": "W", "TYR": "Y", "VAL": "V", "UNK": "X",
}

def is_aa(value):
    return value in STANDARD_RESIDUE_SUBSTITUTIONS

def parse_biopython_structure_mapping(pdb_id, entity, max_seq_len=None):
    chains = Selection.unfold_entities(entity, 'C')
    chains.sort(key=lambda c: c.get_id())
    data = {}

    assert len(chains) == 1, f"Expected one chain in antigen entity for {pdb_id}"

    pos_aa_mapping = {}

    for chain in chains:
        residues = Selection.unfold_entities(chain, 'R')
        residues.sort(key=lambda res: (res.get_id()[1], res.get_id()[2]))

        for res in residues:
            res_id = int(res.get_id()[1])
            if max_seq_len is not None and res_id > max_seq_len:
                break

            resname = res.get_resname()
            if not is_aa(resname):
                continue

            std_name = STANDARD_RESIDUE_SUBSTITUTIONS[resname]
            token = RESIDUE_NAME_TO_TOKEN[std_name]

            pos_aa_mapping[res_id] = token

    return pos_aa_mapping

def check_mutation(args):
    """Check mutation for a single row"""
    idx, pdb_id, mutant, mut_dir = args

    pdb_id = pdb_id.replace('+', '')
    pdb_id = pdb_id.replace('.00', '')

    # 2. update mutate_info
    mutate_infos = mutant
    pdb_path = os.path.join(mut_dir, f"{pdb_id}_{mutate_infos}.pdb")
    # 3. end

    try:
        parser = PDBParser(QUIET=True)
        structure = parser.get_structure(pdb_id, pdb_path)
        pdb_structure = structure[0]

        for mutate_info in mutate_infos.split(','):
            chain_id = mutate_info[1]

            pos_aa_mapping = parse_biopython_structure_mapping(pdb_id, pdb_structure[chain_id])

            # Check mutate info
            if pos_aa_mapping.get(int(mutate_info[2:-1]), '') != mutate_info[-1]:
                error_msg = f"Mutation position mismatch in {pdb_id} for mutation {mutate_info}: expected {mutate_info[-1]}, found {pos_aa_mapping.get(int(mutate_info[2:-1]), '')}, all mutations are {mutate_infos}"
                return error_msg
    except Exception as e:
        error_msg = f"Error processing {pdb_id} for mutation {mutant}: {str(e)}"
        return error_msg

    return None

# 1. read data and set path
data_path = "/home/lfj/projects_dir/MERF/data/ABbind/AB-Bind_645pMulti.csv"
mut_dir = "/home/lfj/projects_dir/MERF/data/ABbind/PDBs_mutated"

data_df = pd.read_csv(data_path)

# 1. end

# 2. parse mutation info
args_list = [
    (idx, data_df.loc[idx, 'pdb_id'], data_df.loc[idx, 'mutant'], mut_dir)
    for idx in range(len(data_df))
]
# 2. end

# 并行处理（使用 tqdm 显示进度）
with Pool(processes=16) as pool:
    results = list(tqdm(
        pool.imap_unordered(check_mutation, args_list),
        total=len(args_list),
        desc="Checking mutations"
    ))

# 收集并打印错误信息
errors = [msg for msg in results if msg is not None]

if errors:
    print(f"\nFound {len(errors)} mutation errors:")
    for error in errors:
        print(error)
else:
    print(f"\nAll {len(data_df)} mutations are correct!")


Checking mutations: 100%|██████████| 1095/1095 [00:08<00:00, 127.45it/s]


Found 2 mutation errors:
Mutation position mismatch in 3NGB for mutation KH52N: expected N, found P, all mutations are IH30T,KH52N,RH53N,GH54S,AH56G,VH57T,RH61Q,PH62K,VH73T,YH74S,YL28S
Mutation position mismatch in 3NGB for mutation KH52N: expected N, found P, all mutations are KH52N
